In [9]:
import moderngl
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from ipycanvas import Canvas, hold_canvas
from pyglm import glm
from plyfile import PlyData
from typing import Any, cast

In [10]:
def splats_from_ply(path):
    vertex = PlyData.read(path)["vertex"]
    nsplats = len(vertex["x"])

    # 3 position + 1 opacity -> GLSL vec4 posopa
    position = np.stack([vertex["x"], vertex["y"], vertex["z"]], axis=1).astype(np.float32)
    opacity = np.asarray(vertex["opacity"], dtype=np.float32).reshape(nsplats, 1)
    posopa = np.concatenate([position, opacity], axis=1)

    # vec4 scale；w 只是用于满足 std430 对齐
    splat_scale = np.stack(
        [
            vertex["scale_0"],
            vertex["scale_1"],
            vertex["scale_2"],
            np.zeros(nsplats, dtype=np.float32),
        ],
        axis=1,
    ).astype(np.float32)

    # PLY 中 rotation 为 (w, x, y, z)，与 shader 当前读取方式一致
    splat_rot = np.stack([vertex[f"rot_{i}"] for i in range(4)], axis=1).astype(np.float32)

    # DC 是第 0 个 SH 系数，形状为 (N, 1, RGB)
    sh_dc = np.stack([vertex[f"f_dc_{i}"] for i in range(3)], axis=1).astype(np.float32)[:, None, :]

    # f_rest 按 RGB 通道分别保存：先恢复 (N, 3, 15)，再转为 (N, 15, 3)
    sh_rest = np.stack([vertex[f"f_rest_{i}"] for i in range(45)], axis=1).astype(np.float32)
    sh_rest = sh_rest.reshape(nsplats, 3, 15).transpose(0, 2, 1)

    # GLSL 是 vec4 shcolor[16]：RGB 后补一个 float，保证每个系数占 16 bytes
    sh_rgb = np.concatenate([sh_dc, sh_rest], axis=1)
    sh_vec4 = np.zeros((nsplats, 16, 4), dtype=np.float32)
    sh_vec4[:, :, :3] = sh_rgb

    # 每条记录：3 个 vec4 + 16 个 vec4 = 76 float32 = 304 bytes
    splats = np.concatenate(
        [posopa, splat_scale, splat_rot, sh_vec4.reshape(nsplats, 64)],
        axis=1,
    )
    assert splats.shape == (nsplats, 76)
    return np.ascontiguousarray(splats, dtype=np.float32)


def get_uniform(shader: Any, name: str) -> moderngl.Uniform:
    return cast(moderngl.Uniform, shader[name])

In [11]:
class SplatRenderer:
    def __init__(self, width, height) -> None:
        self.create_context()
        self.width = width
        self.height = height
        self.canvas = Canvas(width=width, height=height)
        self.vertcode = self.read_shader("splat.vert.glsl")
        self.fragcode = self.read_shader("splat.frag.glsl")
        self.program = self.ctx.program(
            vertex_shader=self.vertcode,
            fragment_shader=self.fragcode,
        )

        self.compcode = self.read_shader("splat.comp.glsl")
        self.compute_program = self.ctx.compute_shader(self.compcode)
        self.sortcode = self.read_shader("splat_sort.glsl")
        self.sort_program = self.ctx.compute_shader(self.sortcode)

        self.fbo = self.ctx.simple_framebuffer((width, height), components=4)
        self.fbo.use()
        self.pixel_buffer = bytearray(self.width * self.height * 4)

        self.orbit_x = 0.0
        self.orbit_y = 0.0
        self.orbit_r = 4.0

        self.update_viewmtx()

        self.near = 0.01
        self.far = 1000
        self.fovy = glm.pi() / 2.0
        self.aspect = self.width / self.height
        self.projmtx = glm.perspective(self.fovy, self.aspect, self.near, self.far)

        self.ctx.point_size = 1.0

    def read_shader(self, path):
        source = ""
        with open(path, "r", encoding="utf-8") as f:
            source = f.read()
        return source

    def set_orbit_x(self, rad):
        self.orbit_x = rad
        self.update_viewmtx()

    def set_orbit_y(self, rad):
        self.orbit_y = rad
        self.update_viewmtx()

    def set_orbit_r(self, radius):
        self.orbit_r = radius
        self.update_viewmtx()

    def update_viewmtx(self):
        x = self.orbit_r * glm.cos(self.orbit_y) * glm.cos(self.orbit_x)
        y = self.orbit_r * glm.cos(self.orbit_y) * glm.sin(self.orbit_x)
        z = self.orbit_r * glm.sin(self.orbit_y)
        self.camera_eye = glm.vec3(x, y, z)
        self.camera_center = glm.vec3(0, 0, 0)
        self.camera_up = glm.vec3(0, 0, 1)
        self.viewmtx = glm.lookAt(self.camera_eye, self.camera_center, self.camera_up)
        return self.viewmtx

    def create_context(self):
        # 服务器 / Docker / 无显示器环境优先尝试 EGL
        try:
            self.ctx = moderngl.create_context(standalone=True, backend="egl")
        except Exception:
            # 本地桌面环境可尝试默认 backend
            self.ctx = moderngl.create_context(standalone=True)

    def setSplatData(self, splats):
        self.splats = np.ascontiguousarray(splats, dtype=np.float32)
        self.nsplats = self.splats.shape[0]
        self.sort_count = 1 if self.nsplats == 0 else 1 << (self.nsplats - 1).bit_length()
        self.input_ssbo = self.ctx.buffer(self.splats.tobytes())
        self.input_ssbo.bind_to_storage_buffer(binding=0)
        output_splats = np.zeros((self.sort_count, 16), dtype=np.float32)
        output_splats[self.nsplats :, 2] = -np.finfo(np.float32).max
        self.ssbo = self.ctx.buffer(output_splats.tobytes())
        self.ssbo.bind_to_storage_buffer(binding=1)
        self.vao = self.ctx.vertex_array(self.program, [])

    def sortSplats(self):
        # 普通 alpha blending 要求 splat 按相机深度从远到近绘制。
        positions = self.splats[:, :3]
        eye = np.array(
            [self.camera_eye.x, self.camera_eye.y, self.camera_eye.z],
            dtype=np.float32,
        )
        center = np.array(
            [self.camera_center.x, self.camera_center.y, self.camera_center.z],
            dtype=np.float32,
        )
        forward = center - eye
        forward_length = np.linalg.norm(forward)
        if forward_length <= np.finfo(np.float32).eps:
            raise ValueError("camera_eye 和 camera_center 不能是同一个位置")
        forward /= forward_length

        # 相机前方的 depth 为正；数值越大表示离相机越远。
        depth = (positions - eye) @ forward
        self.sorted_indices = np.argsort(-depth, kind="stable")
        sorted_splats = np.ascontiguousarray(self.splats[self.sorted_indices], dtype=np.float32)
        return sorted_splats

    def sortSplatsGPU(self):
        if self.nsplats <= 1:
            return

        shader = self.sort_program
        shader["sort_count"].value = int(self.sort_count)

        self.ssbo.bind_to_storage_buffer(binding=0)
        group_count = (self.sort_count + 255) // 256
        k = 2
        while k <= self.sort_count:
            j = k // 2
            while j > 0:
                shader["k"].value = int(k)
                shader["j"].value = int(j)
                shader.run(group_x=group_count)
                self.ctx.memory_barrier()
                j //= 2
            k *= 2

    def set_point_size(self, size):
        self.ctx.point_size = size

    def compute(self):
        if self.nsplats == 0:
            return

        cs = self.compute_program
        get_uniform(cs, "camera.eye").value = (
            self.camera_eye.x,
            self.camera_eye.y,
            self.camera_eye.z,
            1.0,
        )
        get_uniform(cs, "camera.viewmtx").write(self.viewmtx)
        get_uniform(cs, "projection.fovy").value = self.fovy
        get_uniform(cs, "projection.projmtx").write(self.projmtx)
        get_uniform(cs, "viewport.size").value = (float(self.width), float(self.height))
        get_uniform(cs, "nsplats").value = int(self.nsplats)
        self.input_ssbo.bind_to_storage_buffer(binding=0)
        self.ssbo.bind_to_storage_buffer(binding=1)

        group_count = (self.nsplats + 255) // 256
        cs.run(group_x=group_count)

        self.ctx.memory_barrier()
        self.sortSplatsGPU()
        self.ctx.memory_barrier()

    def render(self):
        self.ctx.enable(moderngl.BLEND)
        self.ctx.disable(moderngl.DEPTH_TEST)

        self.ctx.blend_func = (
            moderngl.ONE,
            moderngl.ONE_MINUS_SRC_ALPHA,
        )
        self.ctx.clear(0.08, 0.08, 0.10, 1.0, depth=1.0)

        self.compute()
        self.ssbo.bind_to_storage_buffer(binding=0)
        get_uniform(self.program, "viewport.size").value = (
            float(self.width),
            float(self.height),
        )

        self.vao.render(mode=moderngl.TRIANGLES, vertices=6, instances=self.nsplats)

        self.fbo.read_into(self.pixel_buffer, components=4, alignment=1)
        img = np.frombuffer(self.pixel_buffer, dtype=np.uint8).reshape(self.height, self.width, 4)

        with hold_canvas(self.canvas):
            self.canvas.put_image_data(np.flipud(img))

    def show(self):
        orbitXSlider = widgets.FloatSlider(
            value=0,
            min=-180,
            max=180,
            step=1,
            description="orbit x angle",
            continuous_update=True,
        )

        orbitYSlider = widgets.FloatSlider(
            value=0,
            min=-90,
            max=90,
            step=1,
            description="angle y angle",
            continuous_update=True,
        )

        orbitRadiusSlider = widgets.FloatSlider(
            value=self.orbit_r,
            min=0.1,
            max=10,
            step=0.01,
            description="orbit radius",
            continuous_update=True,
        )

        def on_orbit_x_change(change):
            self.set_orbit_x(glm.radians(change["new"]))
            self.render()

        def on_orbit_y_change(change):
            self.set_orbit_y(glm.radians(change["new"]))
            self.render()

        def on_orbit_r_change(change):
            self.set_orbit_r(change["new"])
            self.render()

        orbitXSlider.observe(on_orbit_x_change, names="value")
        orbitYSlider.observe(on_orbit_y_change, names="value")
        orbitRadiusSlider.observe(on_orbit_r_change, names="value")

        orbitControlBox = widgets.VBox([orbitXSlider, orbitYSlider, orbitRadiusSlider])
        rootBox = widgets.HBox([self.canvas, orbitControlBox])

        display(rootBox)

        self.render()

In [12]:
splats = splats_from_ply("../data/cactus_splat3.ply")
width, height = 1024, 768
renderer = SplatRenderer(width, height)
renderer.setSplatData(splats)
renderer.show()